# County Rainfall and Planting Advisory

**Analyst:** Fidelmah Mbondo (PLP Data Analytics, 1 Million Devs Cohort)

A farmer-support NGO sends SMS advice on planting windows in five counties. I clean the rainfall data, analyse it with NumPy and finish with a short planting advisory. The Machakos row was typed by hand and has errors, so I clean it first.

## Imports and data

I import the libraries I need first, then load the data exactly as given.

I also add a list of month names so I can turn positions into names later.

In [1]:
# --- Imports ---
import csv
import numpy as np

In [2]:
# --- Data, as given in the brief ---
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

# month names, so I can turn a position into a name later
month_names = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]

## Task 1: Clean the Machakos row

The Machakos readings arrived as text.

I wrote clean_rainfall_row() to convert each reading to a whole number inside a try block. It catches only ValueError, which is what Python raises for text like "N/A" or an empty string.

A bad reading is marked as None, then replaced by the rounded average of the valid readings. The function also returns which months were imputed, so nothing is changed silently.

In [3]:
# --- Task 1: clean the Machakos row ---
def clean_rainfall_row(raw_values, months):
    """Convert raw text readings to whole numbers.

    Invalid readings are replaced by the rounded average of the valid ones.
    Returns the cleaned list, the imputed month names and the value used.
    """
    values = []
    for text in raw_values:
        try:
            values.append(int(text.strip()))
        except ValueError:
            values.append(None)          # remember the bad reading for now

    valid = [v for v in values if v is not None]
    fill_value = round(sum(valid) / len(valid))

    imputed = [months[i] for i, v in enumerate(values) if v is None]
    cleaned = [fill_value if v is None else v for v in values]
    return cleaned, imputed, fill_value


machakos_clean, imputed_months, fill_value = clean_rainfall_row(machakos_raw, month_names)

print("Cleaned Machakos row:", machakos_clean)
print("Imputed months:", ", ".join(imputed_months))
print("Value used for imputation:", fill_value, "mm")

Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]
Imputed months: April, September
Value used for imputation: 68 mm


## Task 2: Combine the counties into one array

I add the cleaned Machakos row to the other four rows and build a single 5 x 12 NumPy array: one row for each county and one column for each month. Then I check the shape and the data type.

In [4]:
# --- Task 2: combine into one 5 x 12 array ---
rain = np.array(clean_rows + [machakos_clean])

print("Shape:", rain.shape)
print("Data type:", rain.dtype)
print(rain)

Shape: (5, 12)
Data type: int64
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]


## Task 3: Annual totals and monthly averages

axis=1 collapses across the columns, which gives one total for each county (each row).
axis=0 collapses down the rows, which gives one average for each month across all counties (each column).

The loops below only print the results, because NumPy does the calculations.

In [5]:
# --- Task 3: annual totals and monthly averages ---
annual_totals = rain.sum(axis=1)                  # one total per county
monthly_avg = rain.mean(axis=0).round(1)          # one average per month

for county, total in zip(counties, annual_totals):
    print(f"{county:<9} {total:>5} mm")

print()
for month, avg in zip(month_names, monthly_avg):
    print(f"{month:<10} {avg:>6} mm")

# quick check: the county totals must add up to the grand total
print("\nCheck:", annual_totals.sum() == rain.sum())

Nairobi    1045 mm
Nakuru      916 mm
Kisumu     1371 mm
Mombasa    1075 mm
Machakos    811 mm

January      47.0 mm
February     47.6 mm
March        91.0 mm
April       164.4 mm
May         171.2 mm
June         67.6 mm
July         56.6 mm
August       58.2 mm
September    64.2 mm
October      67.8 mm
November    127.4 mm
December     80.6 mm

Check: True


## Task 4: Wettest month for each county

np.argmax(rain, axis=1) returns the position of the biggest value in each row. A position like 3 means nothing to a reader, so I use it to look up the month name. I use zip() to pair each county with its result.

In [6]:
# --- Task 4: wettest month per county ---
wettest_idx = np.argmax(rain, axis=1)             # position of the biggest value in each row
wettest_month = np.array(month_names)[wettest_idx]
peak_mm = rain.max(axis=1)

for county, month, mm in zip(counties, wettest_month, peak_mm):
    print(f"{county:<9} {month:<9} {mm} mm")

Nairobi   April     219 mm
Nakuru    April     145 mm
Kisumu    April     240 mm
Mombasa   May       240 mm
Machakos  November  182 mm


## Task 5: Dry months

A dry month is one below 50 mm.

rain < 50 makes a table of True and False values (a boolean mask). Each True counts as 1, so summing along axis=1 gives the number of dry months for every county without any loop.

In [7]:
# --- Task 5: dry months (below 50 mm) ---
dry_mask = rain < 50
dry_months = dry_mask.sum(axis=1)

for county, count in zip(counties, dry_months):
    print(f"{county:<9} {count} dry months")

Nairobi   4 dry months
Nakuru    3 dry months
Kisumu    0 dry months
Mombasa   2 dry months
Machakos  6 dry months


## Task 6: Best two-month planting window

rain[:, :-1] is every month except December and rain[:, 1:] is every month except January. Adding them gives the total for each pair of consecutive months (January and February, February and March, and so on).

np.argmax(pairs, axis=1) finds the best pair for every county, and I turn its position into month names.

In [8]:
# --- Task 6: best two-month planting window ---
pairs = rain[:, :-1] + rain[:, 1:]                # each month plus the one after it
best_start = np.argmax(pairs, axis=1)             # first month of the best pair
best_total = pairs.max(axis=1)

planting_windows = [f"{month_names[i]}-{month_names[i + 1]}" for i in best_start]

for county, window, total in zip(counties, planting_windows, best_total):
    print(f"{county:<9} {window:<18} {total} mm")

Nairobi   April-May          395 mm
Nakuru    April-May          265 mm
Kisumu    April-May          420 mm
Mombasa   April-May          390 mm
Machakos  November-December  278 mm


## Task 7: Write the summary file

I build one dictionary for each county and write them with csv.DictWriter, inside a with block so the file always closes. NumPy numbers are converted to plain Python numbers first. Then I read the file back to check it has five rows and the right columns.

In [9]:
# --- Task 7: write the summary CSV ---
summary_rows = [
    {"county": c, "annual_total_mm": int(t), "wettest_month": str(w),
     "dry_months": int(d), "planting_window": p}
    for c, t, w, d, p in zip(counties, annual_totals, wettest_month,
                             dry_months, planting_windows)
]

fieldnames = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(summary_rows)

# read it back to check it
with open("rainfall_summary.csv", newline="") as f:
    rows_read = list(csv.DictReader(f))

print(len(rows_read), "rows written")
for row in rows_read:
    print(row)

5 rows written
{'county': 'Nairobi', 'annual_total_mm': '1045', 'wettest_month': 'April', 'dry_months': '4', 'planting_window': 'April-May'}
{'county': 'Nakuru', 'annual_total_mm': '916', 'wettest_month': 'April', 'dry_months': '3', 'planting_window': 'April-May'}
{'county': 'Kisumu', 'annual_total_mm': '1371', 'wettest_month': 'April', 'dry_months': '0', 'planting_window': 'April-May'}
{'county': 'Mombasa', 'annual_total_mm': '1075', 'wettest_month': 'May', 'dry_months': '2', 'planting_window': 'April-May'}
{'county': 'Machakos', 'annual_total_mm': '811', 'wettest_month': 'November', 'dry_months': '6', 'planting_window': 'November-December'}


## Planting Advisory

For Nairobi, Nakuru, Kisumu and Mombasa, advise farmers to plant in April to May, which is the wettest two-month window in all four counties (265 to 420 mm).

Machakos follows a different pattern, with its best window in November to December (278 mm), so it needs separate messages.

Kisumu is the wettest county overall (1,371 mm) with no dry months, while Machakos is the driest (811 mm) with 6 dry months, so Machakos farmers should also get advice on drought-tolerant crops and saving water.

Nairobi and Nakuru have 4 and 3 dry months, so they should be warned about the dry stretch outside the main rains.

These dates are guidance, not a promise, because the data covers one year and two Machakos readings (April and September) were estimated at 68 mm.

The NGO should confirm the windows against several years of rainfall records before sending final advice.

## Data Quality Notes

- The Machakos row was typed by hand. Of its 12 readings, 10 were valid (one, " 20 ", only had extra spaces and was fixed with strip() ) and 2 were not: "N/A" for April and an empty value for September.
- Both missing readings were replaced with 68 mm, the rounded average of the 10 valid readings (67.5). The imputed values make up 136 mm, or 16.8%, of the Machakos total of 811 mm.
- The Machakos planting window stays November to December with the imputed values. But if the real April reading was above 138 mm, April to May would beat it, so the Machakos advice is the least certain.
- The other four counties were used as given. I did not check them against the original station records.
- The data covers a single year at one station per county, so it cannot show what a normal year looks like.
- A dry month is defined here as below 50 mm. Real planting decisions also depend on the crop, the soil and temperature.